# Topic: NLP: Word Embeddings & Latent Space

## Definition (30-second explanation)
Word embeddings are dense vector representations of words in a continuous, high-dimensional latent space. Algorithms learn these vectors so that words with similar semantic meanings are positioned close together, allowing mathematical operations to capture semantic relationships (e.g., King - Man + Woman = Queen).

## Why Interviewers Ask This
* To check if you understand the transition from traditional NLP (TF-IDF/Bag-of-Words) to modern deep learning NLP.
* To test your grasp of vector spaces, distance metrics (like Cosine Similarity), and their practical implications in search or recommendation engines.
* To ensure you know when to use cheaper static embeddings versus computationally expensive contextual embeddings.

## Core Concepts
* **Latent Space:** A multi-dimensional space where vectors reside. Features aren't manually engineered but learned as hidden (latent) variables.
* **Semantic Mapping:** The geometrical distance between two vectors corresponds to their semantic similarity.
* **Linear Relationships:** Directions in the space encode specific relationships (e.g., gender, verb tense, geography).
* **Static vs Contextual:** Static assigns one fixed vector per word (bank = bank). Contextual generates a vector dynamically based on surrounding words (river bank vs. bank account).

## When to Use
* **Static (Word2Vec, GloVe, FastText):** Low-latency systems, edge devices, resource-constrained environments, or when domain vocabulary is highly specialized but syntax matters less.
* **Contextual (BERT, ELMo, GPT):** Complex intent classification, question answering, semantic search, or when polysemy (multiple meanings of a word) is highly prevalent.

## Advantages
* **Dimensionality Reduction:** Compresses sparse vectors (like one-hot encoding of 100k words) into dense vectors (e.g., 300 dimensions).
* **Generalization:** Models can infer meaning for unseen sentences by composing known word vectors.
* **Transfer Learning:** Pre-trained embeddings can be downloaded and used as features for downstream models with very little data.

## Limitations
* **Out-of-Vocabulary (OOV):** Word2Vec/GloVe struggle with unseen words (FastText partially solves this using subwords).
* **Bias:** Embeddings inherently capture and amplify societal biases present in the training corpus (e.g., associating "doctor" strictly with male pronouns).
* **Resource Intensive:** Contextual embeddings require heavy compute (GPUs) for both training and inference.

## Common Comparisons
* **Word2Vec vs. GloVe:** Word2Vec is predictive (predicts context given word or vice versa using local context windows). GloVe is count-based (factorizes a global word-co-occurrence matrix).
* **Static vs. Contextual:** Static models (Word2Vec) map "apple" to a single vector, ignoring if it's the fruit or the company. Contextual models (BERT) map "apple" to different vectors based on the whole sentence.

## Common Interview Traps
* **Confusing Cosine Similarity with Euclidean Distance:** Euclidean measures absolute magnitude/distance, which is skewed by word frequency. Cosine measures the angle (direction), which isolates semantic similarity regardless of frequency.
* **Assuming Static Models Handle Polysemy:** Thinking Word2Vec can differentiate "bat" (animal) from "bat" (baseball). It cannot; the resulting vector is an average of both contexts.

## Python / SQL Syntax (if applicable)
```python
# Contextual Embeddings via HuggingFace
from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = ["I love my bank account", "I love sitting by the river bank"]
embeddings = model.encode(sentences)

# Compute Cosine Similarity
cosine_scores = util.cos_sim(embeddings[0], embeddings[1])
print(f"Similarity: {cosine_scores.item():.4f}")
```

## Important Formula (if applicable)
- Cosine Similarity measures the cosine of the angle between two vectors:

$$ \text{Cosine Similarity}(A, B) = \frac{A \cdot B}{|A| |B|} $$

(Where 1 means identical direction, 0 means orthogonal/no relation, and -1 means opposite direction).

## 45-Second Interview Answer
"Word embeddings project text into a high-dimensional continuous latent space where semantic meaning is captured geometrically. Static models like Word2Vec learn one fixed vector per word, capturing global semantic relationships and enabling vector arithmetic. However, they fail at polysemy. Modern contextual models like BERT use self-attention to generate dynamic embeddings based on the surrounding sentence, effectively differentiating between a 'river bank' and a 'bank account'. In industry, we use contextual embeddings for accuracy in semantic search, but fall back to static embeddings or FastText when latency and compute are strictly constrained."

## Practice Questions:

### Q1:
**Given a Pandas DataFrame of text queries, apply a mock embedding function and compute the cosine similarity between all pairs strictly using NumPy/Pandas. Return pairs of `query_id`s with a similarity > 0.85.**

In [19]:
# Data:
import pandas as pd
import numpy as np

# A dataset of user queries sent to a customer support bot
df = pd.DataFrame({
    'query_id': [101, 102, 103],
    'query_text': [
        "How do I reset my password?", 
        "I forgot my login password, help me reset it.",
        "Where is the nearest branch?"
    ]
})

# Assume you have a function `get_embedding(text)` that returns a 1D numpy array of shape (384,)
def get_embedding(text):
    # Mocking an embedding vector for the sake of the exercise
    return np.random.rand(384)

In [20]:
df

,query_id,query_text
0,101,How do I reset my password?
1,102,"I forgot my login password, help me reset it."
2,103,Where is the nearest branch?


In [30]:
import pandas as pd
import numpy as np

def find_similar_queries(df, threshold=0.85):
    # 1. Generate embeddings and convert to a 2D NumPy array (Matrix)
    embeddings_list = df['query_text'].apply(get_embedding).tolist()
    matrix = np.array(embeddings_list) 
    
    # 2. Normalize the vectors (Divide each vector by its L2 norm/magnitude)
    # Adding keepdims=True ensures the shape aligns for division
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    #print(norms)
    normalized_matrix = matrix / norms
    #print(normalized_matrix)
    
    # 3. Compute Cosine Similarity via Dot Product of the normalized matrix
    similarity_matrix = np.dot(normalized_matrix, normalized_matrix.T)
    #print(similarity_matrix)
    # 4. Extract pairs above the threshold
    ids = df['query_id'].tolist()
    similar_pairs = []
    
    # Loop through the rows (i)
    for i in range(len(ids)):
        # Start columns (j) at i+1 to avoid self-matches and duplicate pairs
        for j in range(i + 1, len(ids)):
            if similarity_matrix[i, j] > threshold:
                similar_pairs.append((ids[i], ids[j]))
                
    return similar_pairs

In [31]:
find_similar_queries(df= df)

[]

**Interview Tips & Common Traps:**

- The "Normalization" Trick: Interviewers love when you know that Cosine Similarity is just the Dot Product if the vectors are normalized (magnitude of 1). It shows you understand the linear algebra, not just the library APIs.

- The i+1 Loop: Looping for j in range(i + 1, len(ids)) is a classic algorithmic pattern. It prevents checking (0, 0) (self-similarity) and prevents checking (1, 0) if you already checked (0, 1).